# Document Retrieval Engine with TF-IDF

### Tools: Python, scikit-learn, NumPy

This project builds a small **document retrieval engine** using the modular NLP pipeline from Day 10.

The system will:

1. Create a knowledge base containing 20+ text documents.
2. Preprocess and vectorise the documents using the Day 10 `Pipeline` architecture.
3. Store the resulting TF-IDF matrix.
4. Implement `retrieve(query, corpus_matrix, top_k=3)`.
5. Test retrieval with 10 queries.
6. Include at least two ambiguous queries and two completely out-of-domain queries.
7. Analyse retrieval failures.
8. Apply a relevance threshold of `0.1`.
9. Explain vocabulary mismatch and why embeddings are useful.

The notebook is deliberately written as a learning project: new commands and important lines are explained immediately after they are introduced.

## 1. What are we building?

A search engine does not need to understand an entire question like a human.

A simple TF-IDF retrieval system works approximately like this:

```text
Knowledge-base documents
        ↓
Preprocessing
        ↓
TF-IDF vectors
        ↓
Stored TF-IDF matrix
        ↓
                 New query
                     ↓
                 Preprocessing
                     ↓
                 TF-IDF vector
                     ↓
             Cosine similarity
                     ↓
             Rank all documents
                     ↓
              Top K results
```

The important idea is that the documents are converted into numerical vectors **once**. When a user asks a new query, only the query needs to be transformed before comparing it against the stored matrix.

## 2. Why use a knowledge base?

A knowledge base is simply a collection of documents that the retrieval system can search.

For this project, our topic is:

### Personal Finance

The documents cover budgeting, saving, emergency funds, credit cards, loans, investing, retirement, taxes, insurance, and related topics.

Using one coherent topic makes it possible to demonstrate both:

- successful retrieval
- failures caused by vocabulary mismatch or ambiguous wording

## 3. Import the libraries

In [ ]:
import numpy as np

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### New commands explained

- `import numpy as np` loads NumPy and gives it the short name `np`.
- `word_tokenize` splits a sentence into individual tokens.
- `stopwords` gives us common English words such as `the`, `is`, and `and`.
- `PorterStemmer` reduces related word forms to stems.
- `TfidfVectorizer` converts documents into TF-IDF vectors.
- `cosine_similarity` compares the direction of two vectors.

We use the same preprocessing and TF-IDF ideas from the earlier projects.

## 4. Download the NLTK resources used by the Day 10 pipeline

NLTK keeps some language data separate from the Python package itself.

In [ ]:
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

### Why these downloads?

- `punkt` and `punkt_tab` support tokenisation.
- `stopwords` supplies the English stopword list.
- `quiet=True` keeps the notebook output cleaner.

If the resources are already installed, NLTK will not need to download them again.

# Part A — Reuse the Day 10 modular pipeline

The assignment specifically asks us to use the **Day 10 Pipeline class**.

For a standalone submission notebook, the class definitions are included below so the notebook can run independently. The architecture is the same:

```text
PreprocessingModule → VectorizerModule → Pipeline
```

We are reusing that design rather than creating a completely new monolithic retrieval script.

In [ ]:
class PreprocessingModule:
    """Clean raw English text before vectorisation."""

    def __init__(self, remove_stopwords=True, apply_stemming=True):
        self.remove_stopwords = remove_stopwords
        self.apply_stemming = apply_stemming
        self.stop_words = set(stopwords.words("english"))
        self.stemmer = PorterStemmer()

    def transform(self, text):
        if not isinstance(text, str):
            raise TypeError("Input text must be a string.")

        if not text.strip():
            raise ValueError("Input cannot be an empty string.")

        if len(text.strip()) == 1:
            raise ValueError("Single-character queries are not supported.")

        tokens = word_tokenize(text.lower())

        if self.remove_stopwords:
            tokens = [
                token for token in tokens
                if token not in self.stop_words
            ]

        tokens = [
            "".join(character for character in token if character.isalpha())
            for token in tokens
        ]

        tokens = [token for token in tokens if token]

        if not tokens:
            raise ValueError(
                "Input contains no usable alphabetic words."
            )

        if self.apply_stemming:
            tokens = [
                self.stemmer.stem(token)
                for token in tokens
            ]

        return " ".join(tokens)


class VectorizerModule:
    """Fit TF-IDF vectors and calculate cosine similarity."""

    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.corpus_vectors = None
        self.corpus = None

    def fit(self, corpus):
        if not corpus:
            raise ValueError("Corpus cannot be empty.")

        self.corpus = corpus
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)

        return self

    def transform(self, query):
        if self.corpus_vectors is None:
            raise RuntimeError("Call fit(corpus) before transform(query).")

        return self.vectorizer.transform([query])

    def similarity(self, query_vector):
        scores = cosine_similarity(
            query_vector,
            self.corpus_vectors
        )[0]

        return scores

    def rank(self, query_vector):
        scores = self.similarity(query_vector)
        return np.argsort(scores)[::-1]


class Pipeline:
    """Connect preprocessing and vectorisation."""

    def __init__(self, preprocessor=None, vectorizer=None):
        self.preprocessor = preprocessor or PreprocessingModule()
        self.vectorizer = vectorizer or VectorizerModule()

    def fit_corpus(self, corpus):
        """Preprocess and fit the TF-IDF vectorizer on the corpus."""

        if not isinstance(corpus, list):
            raise TypeError("Corpus must be a list of strings.")

        if not corpus:
            raise ValueError("Corpus cannot be empty.")

        cleaned_corpus = [
            self.preprocessor.transform(document)
            for document in corpus
        ]

        self.vectorizer.fit(cleaned_corpus)

        return cleaned_corpus

    def run(self, query, corpus):
        """Run the complete Day 10 workflow and return ranked results."""

        cleaned_corpus = self.fit_corpus(corpus)
        cleaned_query = self.preprocessor.transform(query)

        query_vector = self.vectorizer.transform(cleaned_query)
        scores = self.vectorizer.similarity(query_vector)
        ranked_indices = self.vectorizer.rank(query_vector)

        return [
            {
                "rank": rank,
                "document_index": int(index),
                "document": corpus[index],
                "similarity": float(scores[index])
            }
            for rank, index in enumerate(ranked_indices, start=1)
        ]

## 5. What is new in this Day 10 version?

The main addition is:

```python
fit_corpus(corpus)
```

This is useful for a retrieval engine because the knowledge base should be vectorised **once** and then reused for many queries.

### Why not fit the vectorizer for every query?

Because that would repeat unnecessary work.

Instead:

```text
20+ documents
     ↓
fit once
     ↓
TF-IDF matrix stored
     ↓
Query 1 → compare
Query 2 → compare
Query 3 → compare
...
Query 10 → compare
```

This is much closer to how a real retrieval system is structured.

# Part B — Create the knowledge base

The assignment requires **at least 20 documents**.

We will use 24 documents so that the retrieval engine has enough variety to demonstrate both good and bad matches.

In [ ]:
documents = [
    # Budgeting
    "A monthly budget tracks income and expenses so households can control spending.",
    "The 50 30 20 budgeting rule divides after tax income between needs wants and savings.",
    "Creating a spending plan helps people identify unnecessary expenses and reach financial goals.",

    # Saving and emergency funds
    "An emergency fund provides cash for unexpected expenses such as repairs medical bills or job loss.",
    "Automatic transfers from a checking account to a savings account can make saving money easier.",
    "A high yield savings account may offer more interest than a traditional savings account.",

    # Credit cards
    "Credit card interest can become expensive when a balance is carried from one month to the next.",
    "Paying the full credit card balance by the due date can help avoid interest charges.",
    "A credit score can be affected by payment history credit utilization and the age of credit accounts.",

    # Loans and debt
    "A personal loan provides borrowed money that is repaid through scheduled installments and interest.",
    "Paying down high interest debt can reduce the total amount of interest paid over time.",
    "Student loans may have different repayment options depending on the lender and borrower.",

    # Investing
    "Diversification spreads investment money across different assets to reduce concentration risk.",
    "Index funds track market indexes and can provide broad exposure to many companies.",
    "Investors should consider risk tolerance time horizon and investment goals before choosing assets.",

    # Retirement
    "A retirement account can help people save and invest money for their future financial needs.",
    "Compound growth allows investment returns to generate additional returns over long periods.",
    "Employer retirement plans may include matching contributions that increase an employee's savings.",

    # Taxes
    "Income tax is generally calculated from taxable income after applicable deductions and adjustments.",
    "Tax deductions can reduce taxable income while tax credits can directly reduce the amount of tax owed.",

    # Insurance
    "Health insurance can help cover eligible medical costs in exchange for premiums and other payments.",
    "Home insurance can help protect a property and belongings against certain covered losses.",

    # Financial planning
    "A financial plan can combine budgeting saving investing insurance and long term goals.",
    "Reviewing financial goals regularly helps households adjust their plans when income or expenses change."
]

print("Number of documents:", len(documents))

for index, document in enumerate(documents, start=1):
    print(f"{index:02d}. {document}")

### New commands explained

- `len(documents)` counts the documents.
- `enumerate(documents, start=1)` lets us display document numbers starting at 1.
- `f"{index:02d}"` formats the number with two digits, such as `01`, `02`, and `24`.

We now have a 24-document knowledge base, which satisfies the requirement of at least 20 documents.

# Part C — Preprocess and vectorise the knowledge base

Now we use the Day 10 `Pipeline` class.

The important difference from `pipeline.run()` is that we want to **keep the fitted TF-IDF matrix** because it will be reused for all 10 queries.

In [ ]:
pipeline = Pipeline()

cleaned_documents = pipeline.fit_corpus(documents)

corpus_matrix = pipeline.vectorizer.corpus_vectors

print("Number of documents:", corpus_matrix.shape[0])
print("Number of TF-IDF features:", corpus_matrix.shape[1])
print("TF-IDF matrix shape:", corpus_matrix.shape)

## What is `corpus_matrix`?

The matrix has:

```text
rows    = documents
columns = TF-IDF features
```

So if the output says:

```text
(24, 70)
```

that means:

- 24 document vectors
- 70 learned vocabulary features

The exact number of columns depends on the preprocessing and vocabulary.

### Why store this matrix?

We do not want to rebuild the knowledge base every time someone asks a question.

The matrix is our numerical representation of the searchable knowledge base.

## 6. Inspect one processed document

It is useful to verify what the Day 10 preprocessing actually did.

In [ ]:
print("Original document:")
print(documents[0])

print("\nProcessed document:")
print(cleaned_documents[0])

For example, stopwords may disappear and words may be reduced to stems.

This matters because the TF-IDF vectorizer does not receive the original sentence. It receives the **processed representation**.

# Part D — Build `retrieve(query, corpus_matrix, top_k=3)`

The assignment specifically asks for this function.

The function will:

1. preprocess the query
2. vectorise the query using the already-fitted TF-IDF vectorizer
3. compare it against `corpus_matrix`
4. calculate cosine similarity
5. rank the documents
6. apply the `0.1` relevance threshold
7. return the top K documents and scores

In [ ]:
RELEVANCE_THRESHOLD = 0.1

def retrieve(query, corpus_matrix, top_k=3):
    """Return the top K relevant documents for a query.

    A result is rejected when the highest cosine similarity is
    below RELEVANCE_THRESHOLD.
    """

    cleaned_query = pipeline.preprocessor.transform(query)

    query_vector = pipeline.vectorizer.vectorizer.transform(
        [cleaned_query]
    )

    scores = cosine_similarity(
        query_vector,
        corpus_matrix
    )[0]

    ranked_indices = np.argsort(scores)[::-1]

    highest_score = scores[ranked_indices[0]]

    if highest_score < RELEVANCE_THRESHOLD:
        return {
            "message": "No relevant document found",
            "results": []
        }

    top_indices = ranked_indices[:top_k]

    results = [
        {
            "document_index": int(index),
            "document": documents[index],
            "similarity": float(scores[index])
        }
        for index in top_indices
    ]

    return {
        "message": "Relevant documents found",
        "results": results
    }

## 7. Understand the retrieval function

### Step 1 — preprocess the query

```python
cleaned_query = pipeline.preprocessor.transform(query)
```

The query must go through the **same preprocessing** as the documents.

Otherwise the query and documents would be represented inconsistently.

### Step 2 — vectorise the query

```python
pipeline.vectorizer.vectorizer.transform([cleaned_query])
```

Notice that we use `transform()`, not `fit_transform()`.

The vocabulary was already learned from the knowledge base.

### Step 3 — calculate similarity

```python
cosine_similarity(query_vector, corpus_matrix)
```

This compares one query vector with every document vector.

### Step 4 — rank

```python
np.argsort(scores)[::-1]
```

gives document indices from highest similarity to lowest similarity.

### Step 5 — threshold

```python
if highest_score < RELEVANCE_THRESHOLD:
```

checks whether the best match is still too weak.

If the highest score is below `0.1`, the system refuses to pretend that a low-quality match is relevant.

# Part E — Test the retrieval engine

The assignment requires **10 different queries**.

Our test set contains:

- normal in-domain queries
- two deliberately ambiguous queries
- two completely out-of-domain queries

For evaluation, we also specify an `expected_category`. This allows us to detect when the top result is not the kind of document we expected.

In [ ]:
test_queries = [
    {
        "name": "Q1 - Budgeting",
        "query": "How can I make a monthly spending plan?",
        "expected": "budget"
    },
    {
        "name": "Q2 - Emergency fund",
        "query": "How should I save cash for unexpected expenses?",
        "expected": "emergency"
    },
    {
        "name": "Q3 - Credit card",
        "query": "How do I avoid paying interest on my credit card?",
        "expected": "credit"
    },
    {
        "name": "Q4 - Investing",
        "query": "What strategy spreads investment risk across many companies?",
        "expected": "investing"
    },
    {
        "name": "Q5 - Retirement synonym failure",
        "query": "How can I build a nest egg for my later years?",
        "expected": "retirement"
    },
    {
        "name": "Q6 - Tax",
        "query": "What reduces the amount of income that is taxable?",
        "expected": "tax"
    },
    {
        "name": "Q7 - Ambiguous: planning",
        "query": "How can I make a better financial plan?",
        "expected": "financial planning"
    },
    {
        "name": "Q8 - Ambiguous: protection",
        "query": "How can I protect my family financially?",
        "expected": "ambiguous"
    },
    {
        "name": "Q9 - Out of domain: weather",
        "query": "What will the weather be like tomorrow?",
        "expected": "out of domain"
    },
    {
        "name": "Q10 - Out of domain: cooking",
        "query": "How do I bake a chocolate cake?",
        "expected": "out of domain"
    }
]

print("Number of test queries:", len(test_queries))

## 8. Run all 10 queries

The helper below prints the top three results for each query.

In [ ]:
all_results = {}

for test in test_queries:
    print("=" * 95)
    print(test["name"])
    print("Query:", test["query"])
    print("Expected:", test["expected"])
    print("-" * 95)

    result = retrieve(
        test["query"],
        corpus_matrix,
        top_k=3
    )

    all_results[test["name"]] = result

    print(result["message"])

    if result["results"]:
        for rank, item in enumerate(result["results"], start=1):
            print(
                f"{rank}. "
                f"score={item['similarity']:.3f} | "
                f"document {item['document_index'] + 1}: "
                f"{item['document']}"
            )

    print()

### What should we look for?

For normal in-domain queries, we want the top result to be strongly related to the question.

For ambiguous queries, several different topics may compete.

For out-of-domain queries, we want the threshold to prevent the system from returning a random finance document as though it were relevant.

This is an important difference between:

```text
"give me the mathematically closest document"
```

and:

```text
"give me a document that is actually relevant"
```

A retrieval system needs the second behaviour.

# Part F — Analyse retrieval failures

A retrieval failure happens when the top-ranked document is not the document/topic we expected.

There are several common reasons:

### 1. Vocabulary mismatch

The query uses one word while the document uses another word with a similar meaning.

Example:

```text
Query:  nest egg
Document: retirement savings
```

TF-IDF does not automatically know that those expressions are related.

### 2. Ambiguity

A word such as `plan`, `protect`, or `growth` can refer to multiple concepts.

### 3. Sparse lexical overlap

If the query contains words that rarely appear in the corpus, the vector can have very little overlap with the documents.

### 4. Out-of-domain query

The knowledge base is about personal finance, so a weather or cooking question has no genuinely relevant document.

### 5. Short query

Very short queries provide less vocabulary for TF-IDF to compare.

## 9. Automatically identify possible failures

For this demonstration, we use keyword-based topic labels to inspect whether the top result belongs to the expected topic.

This is an evaluation aid, not part of the retrieval algorithm itself.

In [ ]:
topic_keywords = {
    "budget": ["budget", "spending", "income", "expense", "saving"],
    "emergency": ["emergency", "unexpected", "cash", "saving"],
    "credit": ["credit", "card", "interest", "score"],
    "investing": ["invest", "investment", "fund", "asset", "risk"],
    "retirement": ["retirement", "future", "compound", "employer"],
    "tax": ["tax", "taxable", "deduction", "credit"],
    "financial planning": ["financial", "plan", "goals", "budgeting"],
    "insurance": ["insurance", "protect", "coverage", "premium"]
}

def detect_topic(document):
    text = document.lower()

    topic_scores = {
        topic: sum(keyword in text for keyword in keywords)
        for topic, keywords in topic_keywords.items()
    }

    best_topic = max(topic_scores, key=topic_scores.get)

    if topic_scores[best_topic] == 0:
        return "unknown"

    return best_topic

### New Python commands explained

```python
max(topic_scores, key=topic_scores.get)
```

finds the dictionary key whose value is largest.

For example, if:

```python
{"budget": 3, "credit": 1, "tax": 0}
```

then `max(..., key=...)` returns `"budget"`.

This is only being used to help us inspect our retrieval results.

In [ ]:
failure_analysis = []

for test in test_queries:
    result = all_results[test["name"]]

    if not result["results"]:
        diagnosis = (
            "The query was rejected because its highest similarity "
            "score was below the 0.1 relevance threshold."
        )
        failure_analysis.append(
            (test["name"], "threshold rejection", diagnosis)
        )
        continue

    top_document = result["results"][0]["document"]
    detected_topic = detect_topic(top_document)

    expected = test["expected"]

    if expected == "out of domain":
        diagnosis = (
            "The query is outside the finance knowledge base; "
            "any nonzero lexical overlap can create a weak false match, "
            "which is why the relevance threshold is necessary."
        )
        failure_analysis.append(
            (test["name"], "out-of-domain", diagnosis)
        )

    elif expected == "ambiguous":
        diagnosis = (
            "The wording is ambiguous and overlaps with several finance "
            "topics, so TF-IDF cannot reliably infer the intended meaning."
        )
        failure_analysis.append(
            (test["name"], "ambiguous", diagnosis)
        )

    elif expected not in detected_topic:
        diagnosis = (
            f"The top document was classified as '{detected_topic}' rather "
            f"than the expected '{expected}' topic, indicating lexical "
            "overlap with a competing document."
        )
        failure_analysis.append(
            (test["name"], "retrieval failure", diagnosis)
        )

print("Failure / special-case analysis")
print("=" * 95)

for name, failure_type, diagnosis in failure_analysis:
    print(f"{name} [{failure_type}]")
    print(diagnosis)
    print()

### Why analyse failures?

A good NLP project should not only show successful examples.

Failures teach us what the algorithm can and cannot do.

For each problematic query, we provide a **one-sentence diagnosis**, as required by the assignment.

The important point is to identify a **specific cause**, such as:

- missing vocabulary overlap
- ambiguous wording
- out-of-domain content
- competing documents with stronger word overlap
- threshold rejection

# Part G — Demonstrate vocabulary mismatch directly

The assignment specifically asks us to document how synonym queries can fail.

Let's create a deliberately difficult query:

```text
"How can I build a nest egg for old age?"
```

The knowledge base talks about:

```text
retirement
future
savings
compound growth
```

but it may not contain the exact phrase:

```text
nest egg
old age
```

TF-IDF is based on vocabulary overlap, so a human may understand the meaning while TF-IDF does not.

In [ ]:
synonym_query = "How can I build a nest egg for old age?"

synonym_result = retrieve(
    synonym_query,
    corpus_matrix,
    top_k=3
)

print("Query:", synonym_query)
print(synonym_result["message"])

for rank, item in enumerate(synonym_result["results"], start=1):
    print(
        f"{rank}. score={item['similarity']:.3f} | "
        f"{item['document']}"
    )

## Why can this fail?

TF-IDF does not contain a semantic dictionary.

It does not automatically know:

```text
nest egg ≈ retirement savings
old age ≈ retirement
```

If the words in the query do not occur in the document vocabulary, the corresponding TF-IDF dimensions are zero.

That produces a low similarity score even when a human would consider the document highly relevant.

This is called **vocabulary mismatch**.

# Part H — Relevance threshold

The assignment requires:

> If the highest similarity score across all documents is below `0.1`, return `No relevant document found`.

We implemented:

```python
RELEVANCE_THRESHOLD = 0.1
```

and:

```python
if highest_score < RELEVANCE_THRESHOLD:
    return {
        "message": "No relevant document found",
        "results": []
    }
```

This prevents the system from presenting the least-bad document as if it were genuinely relevant.

In [ ]:
def show_threshold_test(query):
    result = retrieve(query, corpus_matrix, top_k=3)

    print("Query:", query)
    print("Status:", result["message"])

    if result["results"]:
        for item in result["results"]:
            print(
                f"score={item['similarity']:.3f} | "
                f"{item['document']}"
            )

    print()

show_threshold_test("What is the capital of Mars?")
show_threshold_test("How do I make pasta from scratch?")

### Why is the threshold useful?

Without a threshold, a retrieval system is forced to return something even when nothing is relevant.

For example:

```text
Query: How do I make pasta?
```

might still produce a finance document with a tiny score.

That is technically the highest score, but it is not useful.

The threshold lets the system say:

> **No relevant document found**

instead of returning misleading information.

# Part I — Inspect the exact similarity scores for out-of-domain queries

It is useful to see why the threshold matters mathematically.

In [ ]:
def raw_scores(query):
    cleaned_query = pipeline.preprocessor.transform(query)

    query_vector = pipeline.vectorizer.vectorizer.transform(
        [cleaned_query]
    )

    scores = cosine_similarity(
        query_vector,
        corpus_matrix
    )[0]

    return np.sort(scores)[::-1]


for query in [
    "What will the weather be like tomorrow?",
    "How do I bake a chocolate cake?"
]:
    try:
        scores = raw_scores(query)
        print("Query:", query)
        print("Highest score:", round(float(scores[0]), 3))
        print()
    except ValueError as error:
        print("Query:", query)
        print("Preprocessing error:", error)
        print()

### Why inspect the raw score?

The retrieval threshold uses the **maximum similarity score**.

Conceptually:

```text
all document scores
        ↓
take maximum
        ↓
is maximum < 0.1?
        ↓
yes → reject
no  → return top K
```

This is simple and interpretable, although in a production system the threshold would normally be tuned using a labelled validation set.

# Part J — Understand the TF-IDF retrieval limitation

## Vocabulary mismatch

TF-IDF represents a document using dimensions associated with words.

Suppose the vocabulary contains:

```text
retirement
savings
account
investment
```

but a user asks:

```text
How do I build a nest egg?
```

If `nest` and `egg` are not in the relevant documents, there may be little vector overlap.

A human sees:

```text
nest egg → money saved for the future
```

TF-IDF sees:

```text
nest + egg
```

and does not automatically infer the concept.

Therefore:

> **High semantic similarity does not guarantee high TF-IDF similarity.**

This is one of the major limitations of sparse lexical retrieval.

## What does this tell us about embeddings?

Embedding models represent text in a continuous vector space designed to capture meaning.

Instead of representing a sentence mainly as independent word dimensions, an embedding can place related concepts closer together.

Conceptually:

```text
TF-IDF:

"retirement savings"
        ↓
word-weight vector

"nest egg"
        ↓
different word-weight vector
        ↓
possibly low similarity


Embeddings:

"retirement savings"
        ↓
semantic vector

"nest egg"
        ↓
semantic vector
        ↓
potentially high similarity
```

This does not mean embeddings are perfect. They can have their own problems involving ambiguity, domain knowledge, hallucination, bias, and computational cost.

But the comparison teaches an important NLP lesson:

**TF-IDF is excellent for interpretable lexical matching; embeddings are better suited to semantic matching.**

# Part K — Final retrieval engine summary

The complete retrieval workflow is:

```text
24 knowledge-base documents
          ↓
Day 10 preprocessing
          ↓
TF-IDF vectorisation
          ↓
Stored corpus_matrix
          ↓
       New query
          ↓
Same preprocessing
          ↓
Same fitted TF-IDF vocabulary
          ↓
Query vector
          ↓
Cosine similarity against every document
          ↓
Rank scores
          ↓
Highest score < 0.1?
      ↙           ↘
    YES            NO
     ↓              ↓
No relevant       Top K
document          documents
found             + scores
```

## What this project demonstrates

### Retrieval

We can search a knowledge base using a natural-language query.

### Ranking

Documents are sorted by cosine similarity.

### Relevance filtering

Very weak matches can be rejected using a threshold.

### Evaluation

We test multiple queries instead of relying on one example.

### Failure analysis

We identify why retrieval fails.

### NLP limitation

We demonstrate vocabulary mismatch and explain why embeddings can help.

### Reusability

The retrieval system builds on the modular Day 10 architecture rather than duplicating a large block of code.

# Submission checklist

- [x] Knowledge base contains at least 20 text documents.
- [x] Documents stored as a Python list.
- [x] Day 10 preprocessing pipeline reused.
- [x] Documents vectorised with TF-IDF.
- [x] Resulting TF-IDF matrix stored as `corpus_matrix`.
- [x] `retrieve(query, corpus_matrix, top_k=3)` implemented.
- [x] Query is vectorised using the fitted vectorizer.
- [x] Top K documents returned with similarity scores.
- [x] 10 different queries tested.
- [x] At least two ambiguous queries included.
- [x] At least two out-of-domain queries included.
- [x] Retrieval failures / special cases analysed.
- [x] One-sentence diagnoses provided.
- [x] Relevance threshold of `0.1` implemented.
- [x] `"No relevant document found"` response implemented.
- [x] Vocabulary mismatch explained.
- [x] Need for embeddings explained.

## Final takeaway

The project moves from **classification of text processing steps** toward a simple **information retrieval system**.

The central idea is:

> Convert the knowledge base into vectors once, convert each new query into the same vector space, compare the query with every document using cosine similarity, rank the documents, and reject results that are too weak to be considered relevant.